Install Dependencies

In [ ]:
!pip install chess pandas tensorflow zstandard

Download the dataset. We're using lichess evaluations made with Stockfish.

In [ ]:
!wget https://database.lichess.org/lichess_db_eval.jsonl.zst

Decompress the dataset

In [ ]:
import zstandard as zstd
import json

def decompress_zst(input_file, output_file):
    dctx = zstd.ZstdDecompressor()
    with open(input_file, 'rb') as ifh, open(output_file, 'wb') as ofh:
        dctx.copy_stream(ifh, ofh)

decompress_zst('lichess_db_eval.jsonl.zst', 'lichess_db_eval.jsonl')

Preprocess the dataset

In [ ]:
import chess
import numpy as np
import json

def fen_to_vector(fen):
    board = chess.Board(fen)
    vector = np.zeros(64 * 13 + 5 + 10, dtype=np.float32)  # Added 10 new features

    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            piece_type = piece.piece_type - 1
            if piece.color == chess.BLACK:
                piece_type += 6
            vector[square * 13 + piece_type] = 1
        else:
            vector[square * 13 + 12] = 1

    vector[-15] = int(board.turn == chess.WHITE)
    vector[-14] = int(board.has_kingside_castling_rights(chess.WHITE))
    vector[-13] = int(board.has_queenside_castling_rights(chess.WHITE))
    vector[-12] = int(board.has_kingside_castling_rights(chess.BLACK))
    vector[-11] = int(board.has_queenside_castling_rights(chess.BLACK))

    # New features
    vector[-10] = len(list(board.legal_moves))  # Number of legal moves
    vector[-9] = len(list(board.attackers(chess.WHITE, chess.E4)))  # Control of e4 by white
    vector[-8] = len(list(board.attackers(chess.BLACK, chess.E5)))  # Control of e5 by black
    vector[-7] = int(board.is_check())  # Is the current side in check?
    vector[-6] = board.halfmove_clock  # Halfmove clock
    vector[-5] = board.fullmove_number  # Fullmove number
    vector[-4] = len(board.pieces(chess.PAWN, chess.WHITE))  # Number of white pawns
    vector[-3] = len(board.pieces(chess.PAWN, chess.BLACK))  # Number of black pawns
    vector[-2] = int(board.has_castling_rights(chess.WHITE))  # White can castle
    vector[-1] = int(board.has_castling_rights(chess.BLACK))  # Black can castle

    return vector

def preprocess_jsonl_with_evals(jsonl_file, max_games=50000):  # Increased max_games
    positions = []
    evaluations = []
    with open(jsonl_file, 'r') as f:
        for i, line in enumerate(f):
            if i >= max_games:
                break
            game = json.loads(line)
            fen = game['fen']
            evals = game['evals']
            if not evals:
                continue

            evaluation = evals[0]
            cp = evaluation.get('cp')
            mate = evaluation.get('mate')

            if cp is not None:
                normalized_eval = np.tanh(cp / 100.0)  # Use tanh for smoother normalization
            elif mate is not None:
                normalized_eval = np.sign(mate)  # +1 for checkmate, -1 for being checkmated
            else:
                continue

            vector = fen_to_vector(fen)
            positions.append(vector)
            evaluations.append(normalized_eval)

    return np.array(positions), np.array(evaluations)

# Preprocess the data
positions, evaluations = preprocess_jsonl_with_evals('lichess_db_eval.jsonl', max_games=50000)
np.save('positions.npy', positions)
np.save('evaluations.npy', evaluations)

Create and train the model

In [ ]:
!pip install tensorflow

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Conv1D, Flatten, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def create_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(64, kernel_size=3, activation='relu'),
        Conv1D(128, kernel_size=3, activation='relu'),
        Flatten(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dense(64, activation='relu'),
        Dense(1)  # Output layer
    ])

    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model

# Load and preprocess data
positions = np.load('positions.npy')
evaluations = np.load('evaluations.npy')

# Reshape input for Conv1D layers
positions_reshaped = positions.reshape(positions.shape[0], 13, -1)

# Split the data
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(positions_reshaped, evaluations, test_size=0.2, random_state=42)

# Create and train the model
model = create_model(input_shape=(13, positions_reshaped.shape[2]))

# Define callbacks
early_stopping = EarlyStopping(patience=10, restore_best_weights=True)
lr_reducer = ReduceLROnPlateau(factor=0.5, patience=5)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,  # Increased epochs
    batch_size=64,  # Increased batch size
    callbacks=[early_stopping, lr_reducer]
)

# Save the model
model.save('chess_evaluation_model.keras')

Plot training History

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()
